# 🌳 Aula 13 — Árvores e Árvores Binárias
**Disciplina:** Algoritmos e Estrutura de Dados II · FATEC Votorantim (CDN)
**Professor:** Dr. Dilermando Piva Jr. · **Livro:** *Estrutura de Dados com Python* — Cap. 11

**Sumário**
1. Intuição e terminologia
2. Definições formais
3. Árvores binárias: tipos e propriedades matemáticas
4. Implementação 1 — lista de listas
5. Implementação 2 — nós com referências (classe)
6. Percursos: pré-ordem, em-ordem, pós-ordem
7. Árvore Binária de Busca (BST) e remoção de nós
8. Aplicações e resumo

## 1. Intuição: hierarquia no mundo real

Ao contrário de listas (lineares), **árvores representam hierarquia e dependência**. Existe **apenas um caminho possível** entre dois pontos.

| Exemplo | Raiz | Folhas |
|---|---|---|
| Organização corporativa | CEO | Gerentes |
| Relações familiares | Ancestrais | Descendentes |
| Sistema de arquivos | `/` | arquivos (`file1.txt`, `photo.jpg`) |
| Análise sintática (árvore de análise) | Sentença | palavras |

**Onde aparecem na computação:** sistemas operacionais, bancos de dados, redes, compiladores (AST), DOM do HTML, compressão (Huffman) e, em **ciência de dados**, **Árvores de Decisão** e **Random Forest**.

## 2. Terminologia

Uma árvore tem uma **raiz**, da qual partem ramos que ligam **nós** intermediários até as **folhas**.

| Termo | Definição |
|---|---|
| **Nó** | Onde a informação é armazenada |
| **Aresta** | Conexão direta entre dois nós |
| **Raiz** | Único nó sem antecessor (nível 0) |
| **Folha** | Nó sem filhos |
| **Nó interno** | Nó com pelo menos um filho |
| **Pai / Filho / Irmãos** | Irmãos = nós com o mesmo pai |
| **Caminho** | Sequência ordenada de nós ligados por arestas (ex.: a → c → f) |
| **Nível / Profundidade** | Nº de arestas da raiz até o nó (raiz = 0) |
| **Altura da árvore** | Maior profundidade de um nó da árvore |
| **Subárvore** | Um nó e todos os seus descendentes |

> ⚠️ **Cuidado:** nos slides, "altura" aparece de dois jeitos: *altura da árvore* = maior nível; *altura de um nó* = distância máxima até uma folha. Para a árvore toda, dão o mesmo número.

**Exemplo da aula (slide 10):** árvore com raiz `H`

```
Nível 0:           H
                 /   \
Nível 1:        B       F
               / \     / \
Nível 2:      A   C   J   E
                         /|\
Nível 3:                L M N
```

| Propriedade | Valor |
|---|---|
| Nº de nós | 10 |
| Altura | 3 |
| Folhas | A, C, J, L, M, N |
| Nós interiores | H, B, F, E |
| Nós no nível 2 | A, C, J, E |
| Antepassados de E | F, H |
| Descendentes de F | J, E, L, M, N |

> Observação: `E` tem **3 filhos**, então essa é uma árvore **geral**, não binária.

## 3. Definições formais

**Definição 1 (por propriedades)** — árvore = conjunto de nós + conjunto de arestas, com:
1. Existe uma **raiz**: porta de entrada para busca, inserção e remoção.
2. Todo nó, **exceto a raiz**, tem **exatamente um pai**.
3. Existe **um único caminho** da raiz até qualquer nó.
4. Se cada nó tem **no máximo 2 filhos** → **árvore binária**.

**Definição 2 (recursiva)** — uma árvore **ou é vazia**, ou é uma **raiz com zero ou mais subárvores**, cada uma sendo também uma árvore.

> Por isso o código sobre árvores quase sempre é **recursivo**: cada subárvore tem a mesma estrutura da árvore inteira.

## 4. Árvores binárias

- Cada nó tem **grau 0, 1 ou 2** (grau = nº de filhos). Grau 0 = **folha**.
- Os filhos são **esquerdo** e **direito** (a ordem importa).
- **Estritamente binária:** todo nó tem **0 ou 2 filhos**. (Alguns autores chamam isso de "quase completa" e reservam *completa* para árvores com todos os níveis cheios.)

### Tipologia

| Tipo | Características |
|---|---|
| **Completa** | Todos os níveis cheios, exceto possivelmente o último (preenchido da esquerda p/ direita). Ideal para **heaps** e arrays |
| **Perfeita** | Todos os níveis 100% cheios; todas as folhas na mesma altura (caso especial da completa) |
| **Balanceada** | Diferença de altura entre as subárvores de qualquer nó é **≤ 1** (AVL, Rubro-Negra). Crucial para desempenho **O(log n)** |

### Matemática da eficiência

| Propriedade | Fórmula |
|---|---|
| Máx. de nós no nível *l* (raiz = 0) | `2^l` |
| Máx. de nós numa árvore de altura *h* | `2^(h+1) − 1` |
| Folhas × nós internos com 2 filhos (T) | `L = T + 1` |
| Altura mínima para N nós | `h = ⌊log₂ N⌋` (aprox. `log₂ N`) |

Uma árvore balanceada é **muito rasa**, o que acelera a busca.

In [12]:
import math

def max_nos_nivel(l):      return 2 ** l
def max_nos_total(h):      return 2 ** (h + 1) - 1
def altura_minima(n):      return math.floor(math.log2(n))

print("Máx. nós no nível 3:", max_nos_nivel(3))
print("Máx. nós, altura 3 :", max_nos_total(3))
for n in (7, 15, 1_000, 1_000_000):
    print(f"N = {n:>9,} nós -> altura mínima = {altura_minima(n)}")

Máx. nós no nível 3: 8
Máx. nós, altura 3 : 15
N =         7 nós -> altura mínima = 2
N =        15 nós -> altura mínima = 3
N =     1,000 nós -> altura mínima = 9
N = 1,000,000 nós -> altura mínima = 19


## 5. Implementação 1 — **lista de listas**

Formato de cada nó: `[valor, subárvore_esquerda, subárvore_direita]`
- `root[0]` → valor da raiz
- `root[1]` → subárvore esquerda (outra lista)
- `root[2]` → subárvore direita (outra lista)
- `[]` → subárvore vazia

Representação **recursiva**: cada sublista tem a mesma forma da árvore completa.

> 🐛 No slide 22 há erros de digitação no exemplo (`['d', [] []]` sem vírgula e `['e'. [], []]` com ponto). Abaixo está a versão correta.

In [13]:
# Árvore:      a
#             / \
#            b   c
#           / \   \
#          d   e   f     (f é filho DIREITO de c; esquerdo vazio)
my_tree = ['a',
              ['b',
                  ['d', [], []],
                  ['e', [], []]
              ],
              ['c',
                  [],
                  ['f', [], []]
              ]
          ]

print('Raiz              =', my_tree[0])
print('Subárvore esquerda =', my_tree[1])
print('Subárvore direita  =', my_tree[2])

Raiz              = a
Subárvore esquerda = ['b', ['d', [], []], ['e', [], []]]
Subárvore direita  = ['c', [], ['f', [], []]]


> Obs.: o slide desenha `f` como filho de `c` e escreve `['f', [], []], []` — ou seja, `f` à **esquerda**. Eu coloquei à direita só para mostrar o `[]` como posição vazia. Se quiser igualar o slide, troque a ordem dos dois itens da lista de `c`.

### Funções da implementação

In [14]:
def binary_tree(r):
    return [r, [], []]

def insert_left(root, new_branch):
    t = root.pop(1)                      # tira a subárvore esquerda atual
    if len(t) > 1:                       # se existia, ela vira filha ESQUERDA do novo nó
        root.insert(1, [new_branch, t, []])
    else:
        root.insert(1, [new_branch, [], []])
    return root

def insert_right(root, new_branch):
    t = root.pop(2)
    if len(t) > 1:                       # se existia, ela vira filha DIREITA do novo nó
        root.insert(2, [new_branch, [], t])
    else:
        root.insert(2, [new_branch, [], []])
    return root

def get_root_val(root):        return root[0]
def set_root_val(root, val):   root[0] = val
def get_left_child(root):      return root[1]
def get_right_child(root):     return root[2]

**Ponto-chave:** `insert_left/right` **não sobrescrevem** o filho existente. O filho antigo é "empurrado" para baixo, como filho do novo nó.

### Testando a implementação (slide 27)

In [15]:
r = binary_tree(3)
insert_left(r, 4)
insert_left(r, 5)     # 5 entra à esquerda; o 4 desce e vira filho esquerdo de 5
insert_right(r, 6)
insert_right(r, 7)    # 7 entra à direita; o 6 desce e vira filho direito de 7
print(r)

l = get_left_child(r)
print(l)

set_root_val(l, 9)    # l é a MESMA lista dentro de r, então r também muda
print(r)

insert_left(l, 11)
print(r)

[3, [5, [4, [], []], []], [7, [], [6, [], []]]]
[5, [4, [], []], []]
[3, [9, [4, [], []], []], [7, [], [6, [], []]]]
[3, [9, [11, [4, [], []], []], []], [7, [], [6, [], []]]]


### ✏️ Exercício da aula (slide 29)
```python
x = binary_tree('a')
insert_left(x,'b')
insert_right(x,'c')
insert_right(get_right_child(x), 'd')
insert_left(get_right_child(get_right_child(x)), 'e')
```
Desenhe a árvore e dê a representação em lista de listas.

**Resolução passo a passo:**
1. `a` com filho esquerdo `b`
2. `c` à direita de `a`
3. `d` à direita de `c`
4. `e` à esquerda de `d`

```
   a
  / \
 b   c
      \
       d
      /
     e
```

In [16]:
x = binary_tree('a')
insert_left(x, 'b')
insert_right(x, 'c')
insert_right(get_right_child(x), 'd')
insert_left(get_right_child(get_right_child(x)), 'e')
print(x)
assert x == ['a', ['b', [], []], ['c', [], ['d', ['e', [], []], []]]]

['a', ['b', [], []], ['c', [], ['d', ['e', [], []], []]]]


## 6. Implementação 2 — **nós com referências** (classe)

Lista de listas é interessante, mas **pouco intuitiva** quando a árvore cresce: o número de sublistas explode e fica fácil errar.

Alternativa: **encadeamento lógico**, parecido com lista duplamente encadeada. Cada nó guarda **2 referências**: filho esquerdo e filho direito.

| | Lista de listas | Classe com referências |
|---|---|---|
| Estrutura | `[nó, esq, dir]` | objetos ligados por ponteiros |
| Prós | Didática, visualmente aninhada | Encapsula comportamento, escalável, padrão da indústria |
| Contras | Rígida e ilegível em árvores grandes | Um pouco mais de código |

In [17]:
class BinaryTree:
    def __init__(self, valor):
        self.key = valor
        self.left_child = None
        self.right_child = None

    def insert_left(self, valor):
        if self.left_child is None:
            self.left_child = BinaryTree(valor)
        else:
            temp = BinaryTree(valor)             # novo nó
            temp.left_child = self.left_child    # filho antigo desce
            self.left_child = temp

    def insert_right(self, valor):
        if self.right_child is None:
            self.right_child = BinaryTree(valor)
        else:
            temp = BinaryTree(valor)
            temp.right_child = self.right_child
            self.right_child = temp

    def get_right_child(self):  return self.right_child
    def get_left_child(self):   return self.left_child
    def set_root_val(self, v):  self.key = v
    def get_root_val(self):     return self.key

In [18]:
r = BinaryTree('a')
print(r.get_root_val(), r.get_left_child(), r.get_right_child())   # a None None

r.insert_left('b');   print(r.get_left_child().get_root_val())      # b
r.insert_right('c');  print(r.get_right_child().get_root_val())     # c

r.get_left_child().insert_left('d')
print(r.get_left_child().get_left_child().get_root_val())           # d

r.get_left_child().insert_right('e')
print(r.get_left_child().get_right_child().get_root_val())          # e

#        a
#       / \
#      b   c
#     / \
#    d   e

a None None
b
c
d
e


## 7. Percursos (travessias)

Duas regras para memorizar:
1. O filho **esquerdo é sempre visitado antes** do direito.
2. **O nome do percurso diz quando o PAI é visitado:**

| Percurso | Ordem | Pai é visitado... |
|---|---|---|
| **Pré-ordem** (*pre-order*) | Pai → Esq → Dir | **antes** dos filhos |
| **Em ordem** (*in-order*) | Esq → Pai → Dir | **entre** os filhos |
| **Pós-ordem** (*post-order*) | Esq → Dir → Pai | **depois** dos filhos |

### Para que serve cada um

**Pré-ordem**
- Copiar/clonar a estrutura da árvore (começa pela raiz)
- Serializar a árvore para armazenar ou transmitir
- Expressões **prefixas** (notação polonesa): `+ * 3 4 5`
- Sistemas de arquivos: acessar o diretório pai antes dos filhos

**Em ordem**
- Em uma **BST**, visita os valores em **ordem crescente**
- Relatórios ordenados e conversão de árvore em lista ordenada
- Expressões **infixas** (`a + b`)
- Análise em compiladores (AST)

**Pós-ordem**
- **Liberar/apagar** a árvore (filhos antes do pai, evita referência pendente)
- Expressões **posfixas** (notação polonesa reversa): `3 4 * 5 +`
- Calcular **tamanho de diretórios** (soma subpastas antes da pasta pai)
- Geração de código em compiladores

### Implementação (métodos adicionados à classe)

> 🐛 **Erro nos slides (54–56):** o código usa `self.left.preorder()`, mas o atributo se chama `self.left_child`. Do jeito do slide dá `AttributeError`. Corrigido abaixo.

In [19]:
def preorder(self):
    print(self.key)                         # Pai
    if self.left_child:  self.left_child.preorder()    # Esquerda
    if self.right_child: self.right_child.preorder()   # Direita

def inorder(self):
    if self.left_child:  self.left_child.inorder()     # Esquerda
    print(self.key)                         # Pai
    if self.right_child: self.right_child.inorder()    # Direita

def postorder(self):
    if self.left_child:  self.left_child.postorder()   # Esquerda
    if self.right_child: self.right_child.postorder()  # Direita
    print(self.key)                         # Pai

BinaryTree.preorder  = preorder
BinaryTree.inorder   = inorder
BinaryTree.postorder = postorder

### Exemplo do slide 50–53
```
            5
          /   \
        3       8
       / \     / \
      1   4   7   9
       \     /
        2   6
```
Resultados esperados:
- **Pré-ordem:** 5 3 1 2 4 8 7 6 9
- **Em ordem:** 1 2 3 4 5 6 7 8 9 *(ordenado, porque é uma BST)*
- **Pós-ordem:** 2 1 4 3 6 7 9 8 5

In [20]:
t = BinaryTree(5)
t.insert_left(3);  t.insert_right(8)
t.left_child.insert_left(1);   t.left_child.insert_right(4)
t.right_child.insert_left(7);  t.right_child.insert_right(9)
t.left_child.left_child.insert_right(2)
t.right_child.left_child.insert_left(6)

def coletar(no, ordem):
    saida = []
    def visita(n):
        if n is None: return
        if ordem == 'pre':  saida.append(n.key)
        visita(n.left_child)
        if ordem == 'in':   saida.append(n.key)
        visita(n.right_child)
        if ordem == 'post': saida.append(n.key)
    visita(no)
    return saida

print('Pré-ordem :', coletar(t, 'pre'))
print('Em ordem  :', coletar(t, 'in'))
print('Pós-ordem :', coletar(t, 'post'))

assert coletar(t,'pre')  == [5,3,1,2,4,8,7,6,9]
assert coletar(t,'in')   == [1,2,3,4,5,6,7,8,9]
assert coletar(t,'post') == [2,1,4,3,6,7,9,8,5]

Pré-ordem : [5, 3, 1, 2, 4, 8, 7, 6, 9]
Em ordem  : [1, 2, 3, 4, 5, 6, 7, 8, 9]
Pós-ordem : [2, 1, 4, 3, 6, 7, 9, 8, 5]


## 8. Árvore Binária de Busca (BST)

A `BinaryTree` funciona, mas **não é otimizada para busca**. A **Binary Search Tree** implementa o TDA **Map** (mapeia **chave → valor**), usando a estrutura da árvore para buscar de forma eficiente.

| Método | Operação |
|---|---|
| `Map()` | cria mapeamento vazio |
| `put(key, val)` | adiciona par chave-valor (se a chave existe, atualiza o valor) |
| `get(key)` | retorna o valor associado à chave |
| `del map[key]` | remove o par chave-valor |
| `len()` | nº de pares no mapeamento |
| `in` | `True` se a chave pertence ao mapeamento |

### 🔑 Propriedade-chave
> Chaves **menores** que a do pai ficam na subárvore **esquerda**; chaves **maiores**, na **direita**.

Exemplo do slide: inserir **70, 31, 93, 94, 14, 23, 73** (nessa ordem):
```
          70
        /    \
      31      93
     /       /  \
   14      73    94
     \
      23
```

### Remoção de nó — os 3 cenários

| Caso | Situação | O que fazer |
|---|---|---|
| **1** | Nó **folha** (0 filhos) | Simplesmente remove. Mais trivial |
| **2** | Nó com **1 filho** | O filho "sobe" e ocupa o lugar do removido (liga ao avô) |
| **3** | Nó com **2 filhos** | Substituir pelo **sucessor** (mais difícil) |

**Caso 3 em detalhe** (exemplo: remover o `5`):
1. O nó tem ambos os filhos (esq e dir)
2. Percorrer a árvore em busca de um substituto
3. O substituto deve preservar a propriedade da BST dos dois lados
4. Esse nó é a **menor chave que seja maior que a do nó removido**
5. É o **nó sucessor** (o mais à esquerda da subárvore direita)
6. O sucessor tem **no máximo 1 filho** → já sabemos removê-lo (casos 1 e 2). Copia-se `chave` e `valor` dele para o nó atual e remove-se o sucessor

Trecho do slide 72:
```python
if no_atual.tem_ambos_os_filhos():
    sucessor = no_atual.encontrar_sucessor()
    sucessor.remover_sucessor()
    no_atual.chave = sucessor.chave
    no_atual.valor = sucessor.valor
```

### 💡 Implementação complementar (não está nos slides)
Versão compacta, em funções recursivas, para você testar os 3 casos de remoção.

In [21]:
class No:
    def __init__(self, chave, valor=None):
        self.chave, self.valor = chave, valor
        self.esq = self.dir = None

def put(raiz, chave, valor=None):
    if raiz is None: return No(chave, valor)
    if chave < raiz.chave:   raiz.esq = put(raiz.esq, chave, valor)
    elif chave > raiz.chave: raiz.dir = put(raiz.dir, chave, valor)
    else:                    raiz.valor = valor          # chave existe: atualiza
    return raiz

def get(raiz, chave):
    while raiz:
        if chave == raiz.chave: return raiz.valor
        raiz = raiz.esq if chave < raiz.chave else raiz.dir
    return None

def minimo(no):                       # sucessor = mínimo da subárvore direita
    while no.esq: no = no.esq
    return no

def remover(raiz, chave):
    if raiz is None: return None
    if chave < raiz.chave:   raiz.esq = remover(raiz.esq, chave)
    elif chave > raiz.chave: raiz.dir = remover(raiz.dir, chave)
    else:
        if raiz.esq is None: return raiz.dir          # caso 1 (folha) e caso 2 (só dir)
        if raiz.dir is None: return raiz.esq          # caso 2 (só esq)
        suc = minimo(raiz.dir)                        # caso 3: dois filhos
        raiz.chave, raiz.valor = suc.chave, suc.valor # copia dados do sucessor
        raiz.dir = remover(raiz.dir, suc.chave)       # remove o sucessor
    return raiz

def em_ordem(no):
    return em_ordem(no.esq) + [no.chave] + em_ordem(no.dir) if no else []

def altura(no):
    return -1 if no is None else 1 + max(altura(no.esq), altura(no.dir))

In [22]:
raiz = None
for k in (70, 31, 93, 94, 14, 23, 73):          # exemplo do slide 60
    raiz = put(raiz, k)

print('Em ordem:', em_ordem(raiz))              # sempre crescente numa BST
print('Raiz:', raiz.chave, '| esq:', raiz.esq.chave, '| dir:', raiz.dir.chave)
print('Altura:', altura(raiz))

# --- Caso 3 dos slides 62-72: remover o 5 (dois filhos; sucessor = 7) ---
raiz = None
for k in (17, 5, 35, 2, 11, 29, 38, 9, 16, 7, 8):
    raiz = put(raiz, k)
print('\nAntes :', em_ordem(raiz), '| filho esq de 17 =', raiz.esq.chave)
raiz = remover(raiz, 5)
print('Depois:', em_ordem(raiz), '| filho esq de 17 =', raiz.esq.chave)   # agora é o 7

Em ordem: [14, 23, 31, 70, 73, 93, 94]
Raiz: 70 | esq: 31 | dir: 93
Altura: 3

Antes : [2, 5, 7, 8, 9, 11, 16, 17, 29, 35, 38] | filho esq de 17 = 5
Depois: [2, 7, 8, 9, 11, 16, 17, 29, 35, 38] | filho esq de 17 = 7


## 9. Onde árvores são usadas

| Área | Uso |
|---|---|
| Inteligência Artificial / Ciência de Dados | Árvores de Decisão, Random Forest |
| Web | **DOM** do HTML é uma árvore |
| Compressão | **Algoritmo de Huffman** |
| Bancos de dados e redes | **B-Trees** para índices e roteamento |
| Compiladores | AST (árvore de sintaxe abstrata) |

## 10. Resumo rápido

- **Árvore** = estrutura hierárquica, não linear, com 1 raiz e 1 caminho único para cada nó.
- **Árvore binária** = no máximo 2 filhos por nó (esq/dir).
- Nível *l* comporta até `2^l` nós; altura *h* comporta até `2^(h+1) − 1` nós.
- Duas implementações: **lista de listas** (didática) e **classe com referências** (escalável).
- Percursos: o **nome indica quando o pai é visitado** (pré / em / pós).
- **BST**: esquerda < pai < direita. O percurso **em ordem** devolve as chaves ordenadas.
- Remoção na BST: **folha** → apaga; **1 filho** → filho sobe; **2 filhos** → copia o **sucessor** e remove o sucessor.
- Árvores **balanceadas** (AVL, Rubro-Negra) mantêm a altura ≈ `log₂ N` → busca O(log n).